# 00B_FormatData_clean

Format the downloaded USCRN `daily01` text files into a clean long table and a station metadata table.

## What this fixes
The earlier version defined `station_id` using `station_file`, which made the same physical site look like a different station in different years. This notebook defines one `station_id` per physical station using:

- `wban`
- rounded latitude
- rounded longitude

That keeps one site together across all years.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_BASE = Path("uscrn_daily01")
OUT_DIR = Path("uscrn_processed")
OUT_DIR.mkdir(exist_ok=True)

COORD_DECIMALS = 3  # helps avoid tiny float differences splitting one station into multiple IDs

COLS = [
    "WBANNO",
    "LST_DATE",
    "CRX_VN",
    "LONGITUDE",
    "LATITUDE",
    "T_DAILY_MAX",
    "T_DAILY_MIN",
    "T_DAILY_MEAN",
    "T_DAILY_AVG",
    "P_DAILY_CALC",
    "SOLARAD_DAILY",
    "SUR_TEMP_DAILY_TYPE",
    "SUR_TEMP_DAILY_MAX",
    "SUR_TEMP_DAILY_MIN",
    "SUR_TEMP_DAILY_AVG",
    "RH_DAILY_MAX",
    "RH_DAILY_MIN",
    "RH_DAILY_AVG",
    "SOIL_MOISTURE_5_DAILY",
    "SOIL_MOISTURE_10_DAILY",
    "SOIL_MOISTURE_20_DAILY",
    "SOIL_MOISTURE_50_DAILY",
    "SOIL_MOISTURE_100_DAILY",
    "SOIL_TEMP_5_DAILY",
    "SOIL_TEMP_10_DAILY",
    "SOIL_TEMP_20_DAILY",
    "SOIL_TEMP_50_DAILY",
    "SOIL_TEMP_100_DAILY",
]

RENAME = {
    "WBANNO": "wban",
    "LST_DATE": "date",
    "LONGITUDE": "lon",
    "LATITUDE": "lat",
    "T_DAILY_AVG": "tavg",
    "T_DAILY_MAX": "tmax",
    "T_DAILY_MIN": "tmin",
    "RH_DAILY_AVG": "rh_avg",
    "RH_DAILY_MAX": "rh_max",
    "RH_DAILY_MIN": "rh_min",
    "P_DAILY_CALC": "ppt",
    "SOLARAD_DAILY": "solarad",
    "SOIL_MOISTURE_5_DAILY": "sm_5",
    "SOIL_MOISTURE_10_DAILY": "sm_10",
    "SOIL_MOISTURE_20_DAILY": "sm_20",
    "SOIL_MOISTURE_50_DAILY": "sm_50",
    "SOIL_MOISTURE_100_DAILY": "sm_100",
    "SOIL_TEMP_5_DAILY": "st_5",
    "SOIL_TEMP_10_DAILY": "st_10",
    "SOIL_TEMP_20_DAILY": "st_20",
    "SOIL_TEMP_50_DAILY": "st_50",
    "SOIL_TEMP_100_DAILY": "st_100",
}


## Reader and cleaning helpers

In [2]:
def calc_saturation_vapor_pressure_pa(temp_c: pd.Series) -> pd.Series:
    return 611.2 * np.exp((17.67 * temp_c) / (temp_c + 243.5))

def clean_uscrn_values(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    sm_cols = ["sm_5", "sm_10", "sm_20", "sm_50", "sm_100"]
    st_cols = ["st_5", "st_10", "st_20", "st_50", "st_100"]
    temp_cols = ["tavg", "tmax", "tmin"]
    rh_cols = ["rh_avg", "rh_max", "rh_min"]

    # Replace standard missing flags first
    df = df.replace({
        -99999: np.nan,
        -9999: np.nan,
        -999: np.nan,
        -99: np.nan,
    })

    # Basic physical screens
    for c in sm_cols:
        if c in df.columns:
            df.loc[(df[c] < 0) | (df[c] > 1), c] = np.nan

    for c in rh_cols:
        if c in df.columns:
            df.loc[(df[c] < 0) | (df[c] > 100), c] = np.nan

    if "ppt" in df.columns:
        df.loc[df["ppt"] < 0, "ppt"] = np.nan

    # Keep temperatures fairly permissive
    for c in temp_cols + st_cols:
        if c in df.columns:
            df.loc[(df[c] < -80) | (df[c] > 80), c] = np.nan

    return df

def read_uscrn_file(fp: Path) -> pd.DataFrame:
    df = pd.read_csv(
        fp,
        sep=r"\s+",
        header=None,
        names=COLS,
        engine="python",
    )

    df = df.rename(columns=RENAME)

    df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d", errors="coerce")
    df["source_file"] = fp.name
    df["source_stem"] = fp.stem
    df["source_year_dir"] = fp.parent.name

    df = clean_uscrn_values(df)

    # Time helpers
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["doy"] = df["date"].dt.dayofyear

    # Drop Feb 29 to match your old 365-day convention
    is_feb29 = (df["date"].dt.month == 2) & (df["date"].dt.day == 29)
    df = df.loc[~is_feb29].copy()

    # VPD from daily mean T and RH
    es_avg = calc_saturation_vapor_pressure_pa(df["tavg"])
    df["vpd"] = (es_avg * (1 - df["rh_avg"] / 100.0)) / 1000.0

    # Alternative VPD using mean of extrema, closer to your MATLAB script
    tmean_from_extremes = (df["tmax"] + df["tmin"]) / 2.0
    rhmean_from_extremes = (df["rh_max"] + df["rh_min"]) / 2.0
    es_mean = calc_saturation_vapor_pressure_pa(tmean_from_extremes)
    df["vpdmean"] = (es_mean * (1 - rhmean_from_extremes / 100.0)) / 1000.0

    # Rounded coordinates for stable station matching across years
    df["lat_round"] = df["lat"].round(COORD_DECIMALS)
    df["lon_round"] = df["lon"].round(COORD_DECIMALS)

    return df


## Read all station-year files into one long table first

The key step is to concatenate everything **before** defining `station_id`.


In [3]:
all_dfs = []
file_count = 0

for year_dir in sorted(RAW_BASE.iterdir()):
    if not year_dir.is_dir():
        continue
    if not year_dir.name.isdigit():
        continue

    txt_files = sorted(year_dir.glob("*.txt"))
    print(f"{year_dir.name}: {len(txt_files)} files")

    for fp in txt_files:
        try:
            all_dfs.append(read_uscrn_file(fp))
            file_count += 1
        except Exception as e:
            print(f"Failed on {fp}: {e}")

if not all_dfs:
    raise RuntimeError("No USCRN files were read. Check RAW_BASE and the downloaded folder structure.")

uscrn = pd.concat(all_dfs, ignore_index=True)

print("\nFiles read:", file_count)
print("Rows in long table:", len(uscrn))
print("Year span:", int(uscrn["year"].min()), "to", int(uscrn["year"].max()))
uscrn.head()


2000: 2 files
2001: 8 files
2002: 25 files
2003: 45 files
2004: 72 files
2005: 82 files
2006: 97 files
2007: 121 files
2008: 137 files
2009: 155 files
2010: 201 files
2011: 219 files
2012: 222 files
2013: 222 files
2014: 223 files
2015: 153 files
2016: 155 files
2017: 156 files
2018: 157 files
2019: 157 files
2020: 157 files
2021: 157 files
2022: 157 files
2023: 160 files
2024: 159 files
2025: 159 files
2026: 151 files

Files read: 3709
Rows in long table: 1245951
Year span: 2000 to 2026


,wban,date,CRX_VN,lon,lat,tmax,tmin,T_DAILY_MEAN,tavg,ppt,...,source_file,source_stem,source_year_dir,year,month,doy,vpd,vpdmean,lat_round,lon_round
0,53878,2000-11-14,-9.0,-82.56,35.42,NaN,NaN,NaN,NaN,NaN,...,CRND0103-2000-NC_Asheville_13_S.txt,CRND0103-2000-NC_Asheville_13_S,2000,2000,11,319,NaN,NaN,35.42,-82.56
1,53878,2000-11-15,-9.0,-82.56,35.42,NaN,NaN,NaN,NaN,NaN,...,CRND0103-2000-NC_Asheville_13_S.txt,CRND0103-2000-NC_Asheville_13_S,2000,2000,11,320,NaN,NaN,35.42,-82.56
2,53878,2000-11-16,-9.0,-82.56,35.42,NaN,NaN,NaN,NaN,NaN,...,CRND0103-2000-NC_Asheville_13_S.txt,CRND0103-2000-NC_Asheville_13_S,2000,2000,11,321,NaN,NaN,35.42,-82.56
3,53878,2000-11-17,-9.0,-82.56,35.42,NaN,NaN,NaN,NaN,NaN,...,CRND0103-2000-NC_Asheville_13_S.txt,CRND0103-2000-NC_Asheville_13_S,2000,2000,11,322,NaN,NaN,35.42,-82.56
4,53878,2000-11-18,-9.0,-82.56,35.42,NaN,NaN,NaN,NaN,NaN,...,CRND0103-2000-NC_Asheville_13_S.txt,CRND0103-2000-NC_Asheville_13_S,2000,2000,11,323,NaN,NaN,35.42,-82.56


## Define one stable station ID per physical station

This is the main fix.  
Do **not** use `source_file` or `source_stem` to define stations.


In [11]:
station_meta = (
    uscrn[["wban", "lat_round", "lon_round", "lat", "lon"]]
    .drop_duplicates()
    .sort_values(["wban", "lat_round", "lon_round"])
    .reset_index(drop=True)
)

station_meta["station_id"] = np.arange(len(station_meta))

# Keep representative lat/lon per station_id
station_meta = (
    uscrn.groupby("wban", as_index=False)
    .agg(
        lat=("lat", "median"),
        lon=("lon", "median"),
    )
    .sort_values("wban")
    .reset_index(drop=True)
)

station_meta["station_id"] = np.arange(len(station_meta))

uscrn = uscrn.drop(columns=["station_id"], errors="ignore").merge(
    station_meta[["wban", "station_id"]],
    on="wban",
    how="left",
    validate="many_to_one",
)

print("Number of physical stations:", station_meta['station_id'].nunique())
station_meta.head()


Number of physical stations: 237


,wban,lat,lon,station_id
0,3047,31.62,-102.81,0
1,3048,34.36,-106.89,1
2,3054,33.96,-102.77,2
3,3055,36.60,-101.59,3
4,3060,38.54,-107.69,4


## Sanity checks

These should now look sensible:
- number of stations should be on the order of the real USCRN network, not thousands
- rows per station should be large because years are now stacked together


In [12]:
rows_per_station = uscrn.groupby("station_id").size().rename("n_rows")
print(rows_per_station.describe())

years_per_station = (
    uscrn.groupby("station_id")["year"]
    .nunique()
    .rename("n_years")
)
print("\nYears per station:")
print(years_per_station.describe())

# Check whether any WBAN got split into multiple station IDs
wban_station_counts = station_meta.groupby("wban")["station_id"].nunique()
print("\nWBANs split across multiple station IDs:", int((wban_station_counts > 1).sum()))


count     237.000000
mean     5257.177215
std      2983.008837
min       553.000000
25%      1553.000000
50%      6676.000000
75%      7907.000000
max      9255.000000
Name: n_rows, dtype: float64

Years per station:
count    237.000000
mean      15.645570
std        8.248839
min        3.000000
25%        5.000000
50%       20.000000
75%       23.000000
max       27.000000
Name: n_years, dtype: float64

WBANs split across multiple station IDs: 0


## Final cleanup and column order

In [13]:
preferred_order = [
    "station_id", "wban", "date", "year", "month", "doy",
    "lat", "lon", "lat_round", "lon_round",
    "tmax", "tmin", "tavg",
    "ppt", "solarad",
    "rh_max", "rh_min", "rh_avg",
    "vpd", "vpdmean",
    "sm_5", "sm_10", "sm_20", "sm_50", "sm_100",
    "st_5", "st_10", "st_20", "st_50", "st_100",
    "source_file", "source_stem", "source_year_dir",
]

existing = [c for c in preferred_order if c in uscrn.columns]
remaining = [c for c in uscrn.columns if c not in existing]
uscrn = uscrn[existing + remaining].sort_values(["station_id", "date"]).reset_index(drop=True)

uscrn.head()


,station_id,wban,date,year,month,doy,lat,lon,lat_round,lon_round,...,st_100,source_file,source_stem,source_year_dir,CRX_VN,T_DAILY_MEAN,SUR_TEMP_DAILY_TYPE,SUR_TEMP_DAILY_MAX,SUR_TEMP_DAILY_MIN,SUR_TEMP_DAILY_AVG
0,0,3047,2003-05-21,2003,5,141,31.62,-102.81,31.62,-102.81,...,NaN,CRND0103-2003-TX_Monahans_6_ENE.txt,CRND0103-2003-TX_Monahans_6_ENE,2003,1.001,NaN,U,NaN,NaN,NaN
1,0,3047,2003-05-22,2003,5,142,31.62,-102.81,31.62,-102.81,...,NaN,CRND0103-2003-TX_Monahans_6_ENE.txt,CRND0103-2003-TX_Monahans_6_ENE,2003,1.001,20.0,R,NaN,NaN,22.8
2,0,3047,2003-05-23,2003,5,143,31.62,-102.81,31.62,-102.81,...,NaN,CRND0103-2003-TX_Monahans_6_ENE.txt,CRND0103-2003-TX_Monahans_6_ENE,2003,1.001,26.6,R,NaN,NaN,29.4
3,0,3047,2003-05-24,2003,5,144,31.62,-102.81,31.62,-102.81,...,NaN,CRND0103-2003-TX_Monahans_6_ENE.txt,CRND0103-2003-TX_Monahans_6_ENE,2003,1.001,28.5,R,NaN,NaN,27.5
4,0,3047,2003-05-25,2003,5,145,31.62,-102.81,31.62,-102.81,...,NaN,CRND0103-2003-TX_Monahans_6_ENE.txt,CRND0103-2003-TX_Monahans_6_ENE,2003,1.001,21.9,R,NaN,NaN,24.8


## Save outputs

In [14]:
year_min = int(uscrn["year"].min())
year_max = int(uscrn["year"].max())

long_name = f"uscrn_daily_long_{year_min}_{year_max}.parquet"
meta_name = "uscrn_station_metadata.parquet"

uscrn.to_parquet(OUT_DIR / long_name, index=False)
station_meta.to_parquet(OUT_DIR / meta_name, index=False)

print("Saved:")
print(OUT_DIR / long_name)
print(OUT_DIR / meta_name)


Saved:
uscrn_processed/uscrn_daily_long_2000_2026.parquet
uscrn_processed/uscrn_station_metadata.parquet


## Optional xarray export

Only run this if you want a `(station_id, date)` style dataset.


In [15]:
import xarray as xr

vars_keep = [
    "sm_5", "sm_10", "sm_20", "sm_50", "sm_100",
    "rh_avg", "rh_max", "rh_min",
    "tavg", "tmax", "tmin",
    "ppt", "vpd", "vpdmean"
]

wide = (
    uscrn[["station_id", "date"] + vars_keep]
    .drop_duplicates(subset=["station_id", "date"])
    .set_index(["station_id", "date"])
    .sort_index()
)

ds = xr.Dataset.from_dataframe(wide)

meta_indexed = station_meta.set_index("station_id").sort_index()
ds = ds.assign_coords(
    lat=("station_id", meta_indexed.loc[ds.station_id.values, "lat"].values),
    lon=("station_id", meta_indexed.loc[ds.station_id.values, "lon"].values),
    wban=("station_id", meta_indexed.loc[ds.station_id.values, "wban"].values),
)

nc_name = f"uscrn_daily_station_time_{year_min}_{year_max}.nc"
ds.to_netcdf(OUT_DIR / nc_name)

print("Saved:")
print(OUT_DIR / nc_name)


Saved:
uscrn_processed/uscrn_daily_station_time_2000_2026.nc
